# GPT-2 Fine-Tuning on a Car Mechanic Corpus

This notebook fine-tunes a pre-trained GPT-2 model on a custom corpus of car mechanic dialogues using the Hugging Face `transformers` library. The corpus consists of 100 client/mechanic exchanges covering common vehicle symptoms and diagnoses.

The goal is to adapt GPT-2's language generation to this specific domain — given a client description of a problem, the model should generate a plausible mechanic response.

The pipeline covers corpus preparation from PDF, tokenisation, block grouping for causal language modelling, training with the Hugging Face `Trainer`, and text generation with the fine-tuned model.

## Dependencies

Run this cell if you are on Google Colab or a fresh environment.

In [1]:
!pip install torch transformers datasets accelerate pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 12.5 MB/s eta 0:00:00


## Setup

In [2]:
import os
import torch
from pypdf import PdfReader
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline
)

os.environ["TRANSFORMERS_NO_TF"] = "1"

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

torch version: 2.10.0+cu128
CUDA available: True


## Loading the corpus

The corpus is a PDF containing 100 client/mechanic dialogue pairs. We extract the text and save it to a plain text file for the Hugging Face dataset loader.

In [3]:
from google.colab import files
uploaded = files.upload()

Saving mechanic_corpus.pdf to mechanic_corpus.pdf


In [4]:
def extract_text_from_pdf(pdf_path="mechanic_corpus.pdf", out_file="mechanic_corpus.txt"):
    reader = PdfReader(pdf_path)
    all_text = ""
    for page in reader.pages:
        txt = page.extract_text()
        if txt:
            all_text += txt + "\n\n"
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(all_text)
    print(f"Extracted {len(reader.pages)} pages → {out_file}")
    return out_file

corpus_file = extract_text_from_pdf()

Extracted 8 pages → mechanic_corpus.txt


## Preparing the dataset

We load the text file as a Hugging Face dataset, then tokenise it and group the tokens into fixed-size blocks of 1024 — the standard input length for GPT-2.

In [5]:
raw = load_dataset("text", data_files={"train": corpus_file})
print("Examples loaded:", len(raw["train"]))

if len(raw["train"]) == 1:
    single_text = raw["train"][0]["text"]
    paragraphs = [p.strip() for p in single_text.split("\n\n") if p.strip()]
    if len(paragraphs) > 1:
        raw = {"train": Dataset.from_dict({"text": paragraphs})}
        print(f"Split into {len(paragraphs)} examples")

print("Final examples:", len(raw["train"]))

Generating train split: 0 examples [00:00, ? examples/s]

Examples loaded: 321
Final examples: 321


### Tokenisation

In [6]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    return tok(examples["text"])

tokenized = raw["train"].map(
    tokenize_function,
    batched=True,
    num_proc=1,
    remove_columns=["text"]
)
print("Tokenised examples:", len(tokenized))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/321 [00:00<?, ? examples/s]

Tokenised examples: 321


### Block grouping

In [7]:
def group_texts(examples):
    block_size = 1024
    concatenated = []
    for ids in examples["input_ids"]:
        concatenated.extend(ids)
    total_length = len(concatenated)
    if total_length == 0:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    input_ids = [concatenated[i: i + block_size] for i in range(0, total_length, block_size)]
    attention_mask = [[1] * len(ids) for ids in input_ids]
    labels = [ids.copy() for ids in input_ids]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=1,
)
print("Total blocks:", len(lm_dataset))

Map:   0%|          | 0/321 [00:00<?, ? examples/s]

Total blocks: 5


### Train/eval split

In [8]:
if len(lm_dataset) < 2:
    print(f"Dataset too small ({len(lm_dataset)} blocks). Using same set for train and eval.")
    train_dataset = lm_dataset
    eval_dataset = lm_dataset
else:
    split = lm_dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]

print("Train blocks:", len(train_dataset))
print("Eval blocks:", len(eval_dataset))

Train blocks: 4
Eval blocks: 1


## Training

> **Note:** The cells below require a GPU. Run this notebook on Google Colab (Runtime → Change runtime type → T4 GPU).

In [9]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.eos_token_id

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

use_fp16 = torch.cuda.is_available()
print("GPU available:", torch.cuda.is_available())
print("Using fp16:", use_fp16)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPU available: True
Using fp16: True


### Training arguments

In [10]:
training_args = TrainingArguments(
    output_dir="gpt2-mechanic-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    fp16=use_fp16,
    gradient_accumulation_steps=2,
    seed=42,
    report_to="none"
)

print("Training arguments set.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.


### Fine-tuning

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
trainer.train()
trainer.save_model("gpt2-mechanic-finetuned")
print("Done. Model saved to gpt2-mechanic-finetuned/")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting fine-tuning...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.347644
2,No log,3.347644
3,No log,3.310104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Model saved to gpt2-mechanic-finetuned/


## Text generation

Testing the fine-tuned model with a sample prompt.

In [12]:
gen = pipeline(
    "text-generation",
    model="gpt2-mechanic-finetuned",
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

prompt = "CLIENT: My car makes a loud noise when I brake. MECHANIC:"
output = gen(
    prompt,
    max_length=150,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95
)

print("\n--- Generated text ---\n")
print(output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'max_length', 'do_sample', 'num_return_sequences', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated text ---

CLIENT: My car makes a loud noise when I brake. MECHANIC: "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Don't hit me" (a track-clearing sound): "Don't hit me" (a track-clearing sound): "Don't hit me" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Don't hit me" (a track-clearing sound): "Don't hit me" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a track-clearing sound): "Dance" (a
